In [1]:
import pandas as pd
import numpy as np
import glob, os
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)


In [2]:
# Stage 1 — Data intake (processed TON_IoT Network dataset)
# Strategy:
#  - Use stratified random sampling from ALL processed files
#  - Sample 30% from each file to maintain diversity while avoiding OOM
#  - Fallback to raw file only if processed data is unavailable

RAW_PATH = "../data/raw/train_test_network.csv"

PROCESSED_GLOBS = [
    "../data/processed/**/Network_dataset_*.csv",
    "../data/processed/Network_dataset_*.csv",
    "../processed/**/Network_dataset_*.csv",  # fallback
]

# ---------- CONFIG (CHỐT) ----------
SAMPLE_FRAC = 0.30  # Sample 30% from each file (stratified random sampling)
RANDOM_STATE = 42   # For reproducibility
# ----------------------------------

processed_files = []
for pat in PROCESSED_GLOBS:
    processed_files.extend(glob.glob(pat, recursive=True))
processed_files = sorted(list(dict.fromkeys(processed_files)))  # unique + stable

if len(processed_files) > 0:
    print(f"Found {len(processed_files)} processed network files.")
    print(f"Strategy: Sampling {SAMPLE_FRAC*100:.0f}% from each of {len(processed_files)} files")
    print("Loading and sampling...")

    df_list = []
    for i, f in enumerate(processed_files, 1):
        print(f"  [{i}/{len(processed_files)}] {f}")
        df_part = pd.read_csv(f, low_memory=False)
        df_sampled = df_part.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
        df_list.append(df_sampled)

    df = pd.concat(df_list, ignore_index=True)
    data_mode = f"processed_stratified_sample_{len(processed_files)}files_frac{SAMPLE_FRAC}"

else:
    print("No processed Network_dataset_*.csv found -> falling back to raw train_test_network.csv")
    df = pd.read_csv(RAW_PATH)
    data_mode = "raw_single"

print("\nDataset loaded successfully.")
print("Mode:", data_mode)
print(f"Total samples (rows): {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print("Columns:", list(df.columns))


Found 23 processed network files.
Strategy: Sampling 30% from each of 23 files
Loading and sampling...
  [1/23] ../data/processed\Processed_Network_dataset\Network_dataset_1.csv
  [2/23] ../data/processed\Processed_Network_dataset\Network_dataset_10.csv
  [3/23] ../data/processed\Processed_Network_dataset\Network_dataset_11.csv
  [4/23] ../data/processed\Processed_Network_dataset\Network_dataset_12.csv
  [5/23] ../data/processed\Processed_Network_dataset\Network_dataset_13.csv
  [6/23] ../data/processed\Processed_Network_dataset\Network_dataset_14.csv
  [7/23] ../data/processed\Processed_Network_dataset\Network_dataset_15.csv
  [8/23] ../data/processed\Processed_Network_dataset\Network_dataset_16.csv
  [9/23] ../data/processed\Processed_Network_dataset\Network_dataset_17.csv
  [10/23] ../data/processed\Processed_Network_dataset\Network_dataset_18.csv
  [11/23] ../data/processed\Processed_Network_dataset\Network_dataset_19.csv
  [12/23] ../data/processed\Processed_Network_dataset\Networ

In [3]:
# Check label column existence
assert "label" in df.columns, "ERROR: 'label' column not found!"

# Label distribution
label_counts = df["label"].value_counts().sort_index()
label_percent = df["label"].value_counts(normalize=True).sort_index() * 100

label_summary = pd.DataFrame({
    "count": label_counts,
    "percentage (%)": label_percent.round(2)
})

print("Label distribution:")
display(label_summary)


Label distribution:


,count,percentage (%)
label,,
0,239022,3.57
1,6462684,96.43


In [4]:
# Missing values report
# Note: TON_IoT often uses '-' as a placeholder for "not present" in application-level fields.
# We report BOTH:
#  (1) NaN-missing
#  (2) '-' placeholders in object/string columns

# (1) NaN
nan_missing = df.isna().sum()

# (2) '-' placeholder counts (only for object columns)
dash_missing = pd.Series(0, index=df.columns)
obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
for c in obj_cols:
    dash_missing[c] = (df[c].astype(str) == "-").sum()

missing_summary = pd.DataFrame({
    "nan_missing": nan_missing,
    "dash_placeholder": dash_missing,
})
missing_summary["total_missing_like"] = missing_summary["nan_missing"] + missing_summary["dash_placeholder"]
missing_summary["missing_like_%"] = (missing_summary["total_missing_like"] / len(df) * 100).round(4)

missing_summary = missing_summary[missing_summary["total_missing_like"] > 0] \
    .sort_values(by="total_missing_like", ascending=False)

print("Columns with missing values (NaN or '-'):")
display(missing_summary.head(30))

# Duplicates (kept as separate cell below, but we compute here too for convenience)
num_duplicates = int(df.duplicated().sum())
print(f"Number of duplicated rows: {num_duplicates}")


Columns with missing values (NaN or '-'):


,nan_missing,dash_placeholder,total_missing_like,missing_like_%
http_referrer,0,6701705,6701705,100.0000
weird_addl,0,6701376,6701376,99.9951
http_orig_mime_types,0,6701225,6701225,99.9928
weird_notice,0,6700871,6700871,99.9875
weird_name,0,6700871,6700871,99.9875
http_resp_mime_types,0,6699844,6699844,99.9722
ssl_subject,0,6695033,6695033,99.9004
ssl_issuer,0,6695033,6695033,99.9004
ssl_version,0,6694672,6694672,99.8950
ssl_cipher,0,6694672,6694672,99.8950


Number of duplicated rows: 331236


In [5]:
# Duplicated rows (full-row duplicates)
num_duplicates = int(df.duplicated().sum())
dup_pct = round(num_duplicates / len(df) * 100, 4)
print(f"Number of duplicated rows: {num_duplicates} ({dup_pct}%)")

# NOTE: For Stage 1, we keep duplicates as-is to stay faithful to the dataset.
# If needed later, you can de-duplicate ONLY AFTER justifying it in the thesis.


Number of duplicated rows: 331236 (4.9426%)


In [6]:
# Quick schema snapshot (dtype + example + cardinality hints)
schema_snapshot = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "example_value": [df[col].dropna().iloc[0] if not df[col].dropna().empty else None for col in df.columns],
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
})

# For object columns, show how often '-' appears (placeholder)
dash_counts = {}
for col in df.select_dtypes(include=["object"]).columns:
    dash_counts[col] = int((df[col].astype(str) == "-").sum())
schema_snapshot["dash_count"] = [dash_counts.get(c, 0) for c in df.columns]

display(schema_snapshot)


,column,dtype,example_value,n_unique,dash_count
ts,ts,int64,1556025777,315491,0
src_ip,src_ip,object,192.168.1.32,7209,0
src_port,src_port,int64,50266,65536,0
dst_ip,dst_ip,object,192.168.1.186,4758,0
dst_port,dst_port,int64,27228,65536,0
proto,proto,object,tcp,3,0
service,service,object,-,35,5072327
duration,duration,float64,0.0,1536682,0
src_bytes,src_bytes,object,0,21193,0
dst_bytes,dst_bytes,int64,0,17830,0


In [7]:
# Stage 1 Notes + export a one-page dataset summary (Markdown)

# Label is binary
assert "label" in df.columns, "ERROR: 'label' column not found!"

label_counts = df["label"].value_counts().sort_index()
label_percent = (df["label"].value_counts(normalize=True).sort_index() * 100).round(2)
attack_to_normal = float(label_counts.get(1, 0) / max(1, label_counts.get(0, 1)))

summary_lines = []
summary_lines.append("# TON_IoT Processed Network Dataset — Summary (Stage 1)\n")
summary_lines.append(f"- **Loaded mode:** `{data_mode}`\n")
summary_lines.append(f"- **Total samples:** `{len(df)}`\n")
summary_lines.append(f"- **Total columns:** `{df.shape[1]}`\n")
summary_lines.append("\n## Label distribution\n")
summary_lines.append("| label | count | percentage |\n|---:|---:|---:|\n")
for lab in sorted(label_counts.index.tolist()):
    summary_lines.append(f"| {lab} | {int(label_counts[lab])} | {float(label_percent[lab]):.2f}% |\n")
summary_lines.append(f"\n- **Attack/Normal ratio:** `{attack_to_normal:.2f}`\n")

# Missing-like summary (NaN + '-')
nan_missing_total = int(df.isna().sum().sum())
dash_missing_total = int(((df.select_dtypes(include=['object']).astype(str) == '-')).sum().sum()) if len(df.select_dtypes(include=['object']).columns) else 0
summary_lines.append("\n## Missing values\n")
summary_lines.append(f"- **NaN missing (total cells):** `{nan_missing_total}`\n")
summary_lines.append(f"- **'-' placeholders in object columns (total cells):** `{dash_missing_total}`\n")

# Duplicates
num_duplicates = int(df.duplicated().sum())
summary_lines.append("\n## Duplicated rows\n")
summary_lines.append(f"- **Duplicated rows:** `{num_duplicates}` ({num_duplicates/len(df)*100:.2f}%)\n")

# Leakage handling (what we will drop before training)
DROP_COLS_STAGE1 = [c for c in ["src_ip", "dst_ip", "type", "ts"] if c in df.columns]
summary_lines.append("\n## Leakage-prone columns to drop before training\n")
summary_lines.append("- " + ", ".join([f"`{c}`" for c in DROP_COLS_STAGE1]) + "\n")

summary_lines.append("\n## Notes\n")
summary_lines.append("- Stage 1 uses **only** `label` as the training target.\n")
summary_lines.append("- `type` (attack name) is kept **only** for analysis/future work, not for training.\n")
summary_lines.append("- IP addresses are identifiers and can introduce label leakage; they will be removed.\n")

# Write report
REPORT_DIR = "../reports"
os.makedirs(REPORT_DIR, exist_ok=True)
out_path = os.path.join(REPORT_DIR, "dataset_summary.md")
with open(out_path, "w", encoding="utf-8") as f:
    f.writelines(summary_lines)

print("Saved dataset summary to:", out_path)

print("\nBasic dataset notes:")
print("- Label is binary (0 = normal, 1 = attack)")
print("- TON_IoT contains numeric, categorical, and placeholder '-' values (treated as missing-like for reporting)")
print("- No ML preprocessing has been applied in this notebook (Stage 1 training is done in the next notebook)")
print("- Leakage-prone columns (IP/type/timestamp) will be handled explicitly before training")


Saved dataset summary to: ../reports\dataset_summary.md

Basic dataset notes:
- Label is binary (0 = normal, 1 = attack)
- TON_IoT contains numeric, categorical, and placeholder '-' values (treated as missing-like for reporting)
- No ML preprocessing has been applied in this notebook (Stage 1 training is done in the next notebook)
- Leakage-prone columns (IP/type/timestamp) will be handled explicitly before training
